# Cleaning Prompt Testing

In [ ]:
import ast
from tqdm import tqdm

from language_model_service_api.languagemodelservice_api_embeddings_v3 import GenericEmbeddingsRequest
from language_model_service_api.languagemodelservice_api_completion_v3 import GptChatCompletionRequest
from language_model_service_api.languagemodelservice_api import ChatMessage, ChatMessageRole

from palantir_models.models import GenericEmbeddingModel
from palantir_models.models import OpenAiGptChatLanguageModel

from string import Template

class PromptTemplate(Template):
    delimiter = ""

model = OpenAiGptChatLanguageModel.get("GPT_4o")

In [ ]:
topic_list_inputs = [
    ["topic_1"],
    ["TOPIC X"],
    ["Increase Communication"],
    ["A","B"],
    ["Improve leadership", "Send out letters"],
    ["Improve leadership", ["Send out letters"]],
    [["Improve leadership"], ["Send out letters"]],
    "Of course! Here is your new list: ['Topic A','Topic B']",
    "Of course! Here is your new list: [['Topic A'],['Topic B']]",
    "python:['A','B']",
    "python:['A',['B']]",
    ["A", ["B","C"]],
    ["Improve Communication", "Increase Communication"],
    [[["Longer lunch breaks"]]],
    'Certainly. ["Topic A"], ["Topic B"], "C"]'
    
]

topic_list_outputs = [
    ["topic_1"],
    ["TOPIC X"],
    ["Increase Communication"],
    ["A","B"],
    ["Improve leadership", "Send out letters"],
    ["Improve leadership", "Send out letters"],
    ["Improve leadership", "Send out letters"],
    ["Topic A","Topic B"],
    ["Topic A","Topic B"],
    ["A","B"],
    ["A","B"],
    ["A","B","C"],
    ["Improve Communication", "Increase Communication"],
    ["Longer lunch breaks"],
    ["Topic A", "Topic B", "C"]
]

In [ ]:
cleaning_prompt = PromptTemplate("""Clean a string containing a list of topics to be a flat python list.
Your response must strictly be a flat python list containing all of the topics, with no additional information.

Example 1:
If the input is: [["Topic 1"], ["Topic 2"]]
Your output must be: ["Topic 1", "Topic 2"]

Example 2:
If the input is: [["Topic 1", 'Topic 2'],"Topic 3"]
Your output must be: ["Topic 1", "Topic 2", "Topic 3"]

Example 3:
If the input is: [["Topic 1", "Topic 2", "Topic 3", "Topic 4"], "Topic 5", "Topic A", ]]
Your output must be: ["Topic 1", "Topic 2", "Topic 3", "Topic 4", "Topic 5", "Topic A"]

Example 4:
If the input is: ["COMMUNICATION", "IMPROVE LEADERSHIP"]
Your output must be: ["COMMUNICATION", "IMPROVE LEADERSHIP"]

Example 5: Certainly, here is the cleaned list you requested: ["Increase the amount of staff training", "More AL"]
Your output must be: ["Increase the amount of staff training", "More AL"]

Example 6: ["better education", "more oppertunities for progression"] - if you need any more help, please feel free to ask.
Your output must be: ["better education", "more oppertunities for progression"]

Rules:
- Do not include any commentary or markdown tags in your response.
- Do not include a variable assignment.
- All topic spelling must remain the same, do not change or correct any spelling.
- Your response should be valid python list syntax only - that is, starting with an opening square bracket, then a comma-separated list of the topics, each in quotation marks, and a closing square bracket.

The string you must simplify is: {response}""")

In [ ]:
def get_response(test_input):
    formatted_cleaning_prompt = cleaning_prompt.safe_substitute(response=test_input)
    response = model.create_chat_completion(GptChatCompletionRequest([ChatMessage(ChatMessageRole.USER, formatted_cleaning_prompt)]))
    raw_content = response.choices[0].message.content  # Extract the raw content
    return raw_content

In [ ]:
error_count = 0
error_messages = []

for LLM_input, expected_output in tqdm(zip(topic_list_inputs, topic_list_outputs), desc="Testing..."):
    response = get_response(LLM_input)
    
    try:
        topic_array = ast.literal_eval(response)
    except:
        print("WILL NOT COMPILE")
        
    match = all([x == y for x, y in zip(topic_array, expected_output)])
    if not match:
        error_messages.append(f"Failed with input: '{LLM_input}' and output: '{topic_array}'")
        error_count += 1
                 
print(f"Done with {error_count} error(s).")
for m in error_messages:
    print(m)